In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

/var/folders/6c/k1rj1g954qjd9rc6f7j_ht3w0000gn/T/ipykernel_45593/715490466.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
loader = PyPDFLoader("../data/medical_report.pdf")
doc = loader.load()

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitter_data = splitter.split_documents(doc)

In [6]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [7]:
vector_store =  Chroma.from_documents(
    documents=splitter_data,
    embedding=embeddings
)

#### Chain - Context_generator | prompt | llm | str-parser

In [9]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [10]:
def get_context(query: str):
    data = vector_store.similarity_search(query=query)

    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {
        "context":context,
        "question":query
    }

In [11]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answer based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context: {context}
    Question: {question}
""")

In [12]:
rag_chain = get_context | prompt | llm

In [13]:
res = rag_chain.invoke("Collected Date and time of this report? ")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [14]:
print(res.content[0]["text"])

Based on the context provided, the collected date and time of the report is:
**9/7/2025 4:56:00PM** (or **09/07/2025 04:56:00PM**).
